
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020FFD4B4EC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020FFD4B5BE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020FFD4B4EC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020FFD4B5BE0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out

In [6]:
for chunk in model.stream("Provide details about the moview Inception"):
    print(chunk)

content='' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019e6d6b-c68d-7a93-9472-c073b810d261' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='<think>' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019e6d6b-c68d-7a93-9472-c073b810d261' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='\n' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019e6d6b-c68d-7a93-9472-c073b810d261' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='Okay' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019e6d6b-c68d-7a93-9472-c073b810d261' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=',' additional_kwargs={} response_metadata={'model_provider': 'groq'} id='lc_run--019e6d6b-c68d-7a93-9472-c073b810d261' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content=' so' additional_kwargs={} response_metadata={'model_provider':

### To retreive only the required output

In [ ]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure (`include_raw = True`)

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. The title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it an 8.8/10. So I should structure the tool call with those parameters. Make sure the types are correct: title and director as strings, year as an integer, and rating as a number. No missing required fields. Alright, that should cover it.\n', 'tool_calls': [{'id': 'ykp2d0zkt', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 210, '

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]       # Can have multiple actors
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")    # By default None

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Race3")
response

MovieDetails(title='Race 3', year=2018, cast=[Actor(name='Varun Dhawan', role='Arjun'), Actor(name='Alia Bhatt', role='Pooja'), Actor(name='John Abraham', role='Rohit'), Actor(name='Anil Kapoor', role='Panditji'), Actor(name='Eshan Natarajan', role='Arjun (young)'), Actor(name='Shiney Ahuja', role='Arjun (middle age)')], genres=['Action', 'Thriller', 'Drama'], budget=15.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie Dhoom2")
response

{'director': 'Farhan Akhtar', 'rating': 7.5, 'title': 'Dhoom 2', 'year': 2006}

# CRAZY

In [14]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Joss Whedon', 'role': 'Director'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'The Avengers',
 'year': 2012}

In [15]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9862b849-b4e5-424c-8065-5e457907fad6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". Let me see. The function provided is called ContactInfo, and it requires name, email, and phone. \n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function\'s parameters don\'t specify a particular format, so I\'ll assume that the user provided it correctly. \n\nI need to make sure all required fields are present. The name, email, and phone are all there. Now, I\'ll structure the JSON object with these parameters. The name is "John Doe", email is "john@example.co

In [18]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [20]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [21]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')